In [3]:
#!/mrhome/amingk/anaconda3/envs/7tpd/bin/python

import numpy as np 
import pandas as pd
import stan
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.append('/mrhome/amingk/Documents/7TPD/ActStimRL')
from Madule import utils
import nest_asyncio
import os

In [4]:
# Main directory of the subject
readMainDirec = '/mnt/projects/7TPD/bids/derivatives/fMRI_DA/AllBehData/'
# read collected data across all participants
behAll = pd.read_csv(f'{readMainDirec}/NoNanBehAll.csv')


In [5]:
# Find chosen amount for each trial
chosenAmount = behAll['leftChosen']*behAll['winAmtLeft'] + (1-behAll['leftChosen'])*behAll['winAmtRight'] 
# Calculate the probability of high amount is chosed or lower amount
behAll['chosenHighWinAmt'] = chosenAmount>=50
# gtoup label 1,2 3 are PD-OFF, HC and PF-ON respectively
behAll['group'] = behAll['group'].replace([1,2,3], ['PD-OFF', 'HC', 'PD-ON']) 

 
"""biases for each participant"""
bias_seperate_run= behAll.groupby(['sub_ID','session', 'run', 'block', 'phase'], as_index=False)[['leftChosen', 'chosenHighWinAmt', 'pushed', 'yellowChosen']].mean()

In [6]:
bias_seperate_run

,sub_ID,session,run,block,phase,leftChosen,chosenHighWinAmt,pushed,yellowChosen
0,sub-004,1,1,Act,phase1,0.600000,0.500000,0.400000,0.700000
1,sub-004,1,1,Act,phase2,0.454545,0.727273,0.545455,0.272727
2,sub-004,1,1,Act,phase3,0.384615,0.846154,0.692308,0.461538
3,sub-004,1,1,Stim,phase1,0.315789,0.368421,0.526316,1.000000
4,sub-004,1,1,Stim,phase2,0.523810,0.714286,0.619048,0.809524
...,...,...,...,...,...,...,...,...,...
895,sub-121,2,2,Act,phase2,0.285714,0.642857,0.142857,0.571429
896,sub-121,2,2,Act,phase3,0.571429,0.714286,0.714286,0.642857
897,sub-121,2,2,Stim,phase1,0.642857,0.500000,0.285714,0.142857
898,sub-121,2,2,Stim,phase2,0.642857,0.571429,0.428571,0.785714


In [7]:
# get the larger probability for each feature
bias_seperate_run['left_right'] = np.maximum(bias_seperate_run['leftChosen'], 1-bias_seperate_run['leftChosen'])
bias_seperate_run['chosenHighWinAmt'] = bias_seperate_run['chosenHighWinAmt']
bias_seperate_run['action'] = np.maximum(bias_seperate_run['pushed'], 1-bias_seperate_run['pushed'])
bias_seperate_run['color'] = np.maximum(bias_seperate_run['yellowChosen'], 1-bias_seperate_run['yellowChosen'])


In [8]:
# mean across subject in each condition
bias_mean_individual = bias_seperate_run.groupby(['sub_ID', 'block'], as_index=True)[['left_right', 'chosenHighWinAmt', 'action', 'color']].mean()

In [9]:
bias_mean_individual[bias_mean_individual['left_right']>.80]

,,left_right,chosenHighWinAmt,action,color
sub_ID,block,,,,


In [15]:
bias_mean_individual[bias_mean_individual['chosenHighWinAmt']>.80]

left_right  chosenHighWinAmt    action     color
sub_ID  block                                                  
sub-030 Act      0.612268          0.979494  0.651250  0.614482
        Stim     0.604919          0.980159  0.648325  0.581214
sub-065 Act      0.578258          0.819801  0.701122  0.609054
sub-067 Act      0.607364          0.810460  0.631545  0.608452
        Stim     0.595799          0.800514  0.603610  0.656274
sub-069 Act      0.576374          0.860806  0.637729  0.652564
        Stim     0.619524          0.851905  0.629286  0.597857
sub-108 Act      0.590375          0.826533  0.664078  0.583280
        Stim     0.568177          0.882533  0.613460  0.620927

In [14]:
bias_mean_individual[bias_mean_individual['action']>.80]

,,left_right,chosenHighWinAmt,action,color
sub_ID,block,,,,
sub-012,Act,0.587894,0.493297,0.878663,0.623919
sub-042,Act,0.572472,0.634097,0.825456,0.587001
sub-048,Act,0.598156,0.654261,0.808761,0.628376
sub-080,Act,0.550312,0.520910,0.843132,0.645601


In [13]:
bias_mean_individual[bias_mean_individual['color']>.80]

,,left_right,chosenHighWinAmt,action,color
sub_ID,block,,,,
sub-048,Stim,0.591630,0.635227,0.578961,0.851503
sub-079,Stim,0.620220,0.486447,0.580934,0.857619
sub-080,Stim,0.629937,0.476920,0.613062,0.810495
sub-088,Stim,0.597856,0.513019,0.570468,0.830033
